#网络架构

![softmax回归是一种单层神经网络](../resource/pytorch/img/softmaxreg.svg)
:label:`fig_softmaxreg`

# softmax回归的从零开始实现

## 导包和数据集构建

In [4]:
import torch
from IPython import display
from d2l import torch as d2l

In [5]:
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)
for X, y in train_iter:
    break
X.shape

torch.Size([256, 1, 28, 28])

## 初始化模型参数
原始数据集中的每个样本都是$28 \times 28$的图像。
本节**将展平每个图像，把它们看作长度为784的向量作为输入，则输入的形状为(batch_size, 784)**



权重将构成一个$784 \times 10$的矩阵，
偏置将构成一个$1 \times 10$的行向量。

In [ ]:
num_input = 784
num_output = 10

W = torch.normal(0, 0.01, size=(num_input, num_output), requires_grad=True)
b = torch.zeros(size=(1, 10), requires_grad=True)


## 定义softmax操作

表达式

$$
\mathrm{softmax}(\mathbf{X})_{ij} = \frac{\exp(\mathbf{X}_{ij})}{\sum_k \exp(\mathbf{X}_{ik})}.
$$

[**实现softmax**]由三个步骤组成：

1. 对每个项求幂（使用`exp`）；
1. 对每一行求和（小批量中每个样本是一行），得到每个样本的规范化常数；
1. 将每一行除以其规范化常数，确保结果的和为1。

要注意的是式中的X其实是XW + b的结果，是一个 形状(batch_size, 10)的矩阵，因此每一行都有10个元素(每一行是一个样本)

In [ ]:
def softmax(X):
    X_exp = torch.exp(X)
    line_exp = torch.exp(X.sum(1, keepdims = True))
    return X_exp / line_exp #line_exp的每一行元素都广播成同一元素

## 定义模型

**实现softmax回归模型**

教材上说的展平为向量，实际上是一个二维的(batch_size, 1)列向量

In [ ]:
def net(X):
    return softmax(torch.matmul(X.reshape(-1, W.shape[0]), W) + b ) #得把数据集reshape成(batch_size, num_input)形状
    #y_hat shape:[batch_size, num_output]

## 定义损失函数

交叉熵损失函数

$$ l(\mathbf{y}, \hat{\mathbf{y}}) = - \sum_{j=1}^q y_j \log \hat{y}_j. $$

In [ ]:
y = torch.tensor([0,2])
y_hat = torch.tensor([[1,2,3,4], [5,6,7,8]])
y_hat[[0, 1], y]

TypeError: list indices must be integers or slices, not tuple

In [ ]:
def cross_entropy(y_hat, y):
    return - torch.log(y_hat[range(len(y_hat)), y]) * y

cross_entropy(y_hat, y)

tensor([-0.0000, -1.9459])

## 分类精度

即正确预测数量与总预测数量之比。

In [16]:
a = torch.randn(4, 4)
print(a.type(dtype=torch.int32))


tensor([[ 2,  0,  0, -1],
        [ 1,  0,  1,  0],
        [ 0,  0,  0,  0],
        [ 1,  0,  0,  0]], dtype=torch.int32)


In [6]:
def accuracy(y_hat, y):
    """计算单样本中预测正确的数量"""
    y_hat_argmax = y_hat.argmax(dim=1)
    cmp = y_hat_argmax.type(y.dtype) == y
    return float(cmp.sum())   #要用float把tensor形式转化为python的float形式

In [8]:
y = torch.tensor([0, 2])
y_hat = torch.tensor([[0.1, 0.3, 0.6], [0.3, 0.2, 0.5]])
print(accuracy(y_hat, y) / len(y))

0.5


定义一个实用程序类`Accumulator`，用于对多个变量进行累加

在(**`Accumulator`实例中创建了2个变量，
分别用于存储正确预测的数量和预测的总数量**)。
当我们遍历数据集时，两者都将随着时间的推移而累加。

In [ ]:
class Accumulator:
    def __init__(self, n):
        self.data = [0.0] * n            
    
    def add(self, *args):
        self.data = [a + float(b) for a,b in zip(self.data, args)]  #关键：zip配对
        
    def __getitem__(self, idx):
        self.data[idx] = self[idx]

In [ ]:
acc = Accumulator(2)
acc.add(2.0,3.0)

: 

创建一个通用函数，对于任意数据迭代器`data_iter`可访问的数据集，可用于评估在任意模型`net`的精度

用到上面创建的Accumulator类

In [ ]:
def evaluate_accuracy(net, data_iter):
    """计算在指定数据集上模型的精度"""
    # 将模型设置为评估模式
    if isinstance(net, torch.nn.Module):
        net.eval()
    # Accumulator类储存正确预测数、预测总数
    accum = Accumulator(2)
    # 累加所有数据集的正确预测数、预测总数
    with torch.no_grad():   #因为调用了神经网络实例net，所以禁用梯度计算
        for X, y in data_iter:
            accum.add(accuracy(net(X), y), y.numel()) #不能用len(y)
    
    return accum[0] / accum[1]

## 训练